# Expression handling in Python interface

The intent is to add an expression handler to the DISPATCH Python interface, where one can register ad hoc expressions to perform with the known 3-D variables as input.  

Any variable ``v`` in the RHSs of the expressions that is not also in one of the LHSs is assumed to be evaluated as ``p.var(v)``

## Test setup

In [1]:
%pylab

Using matplotlib backend: Qt5Agg
Populating the interactive namespace from numpy and matplotlib


In [2]:
cd C:\cygwin64\home\codes\dispatch\experiments\stellar_atmospheres\python

C:\cygwin64\home\codes\dispatch\experiments\stellar_atmospheres\python


In [3]:
import dispatch
sn=dispatch.snapshot(2)
p=sn.patches[0]
p.var('d').shape

(20, 20, 20)

In [8]:
p.keys

{'letters': ['d',
  'e',
  'et',
  's',
  'px',
  'py',
  'pz',
  'bx',
  'by',
  'bz',
  'qr',
  'tt',
  'phi',
  'p1',
  'p2',
  'p3',
  'b1',
  'b2',
  'b3'],
 'numbers': [0, 4, -1, 4, 1, 2, 3, 5, 6, 7, -1, -1, -1, 1, 2, 3, 5, 6, 7],
 'expressions': ['ux', 'uy', 'uz', 'u1', 'u2', 'u3', 'ee', 'E'],
 'aux': dict_keys(['rkap', 'src', 'tt', 'qr'])}

## Expression class

In [63]:
class Expressions(object):
    def __init__(self,patch=None,verbose=0):
        """ Initialize expressions """
        self.verbose=verbose
        self.v={}
        self.ops=' +-*/()'
        self.known=[]
        self.unknown=[]
        self.globals=globals()
        self.pvars=[]
        if not patch is None:
            for k,v in patch.keys.items():
                if k != 'numbers':
                    self.pvars.extend(v)
                    self.known.extend(v)
            print('patch variables:',self.pvars)
        print('EXPRESSIONS:')

    def register(self,expression):
        """ Register expressions """
        print(' ',expression)
        lhs,rhs=expression.split('=')
        self.v[lhs]=rhs
        self.unknown.append(lhs)

    def split(self,rhs):
        """ Split a RHS into words, based on operators in self.ops """
        words=[]
        def add(w):
            if w != '' and w not in words:
                words.append(w) 
        word=''
        for c in rhs:
            if c in self.ops:
                add(word)
                word=''
            else:
                word+=c
        add(word)
        return words

    def classify(self,verbose=0):
        """ Classify RHS words into known and unknown """
        words=[]
        for rhs in self.v.values():
            words.extend(self.split(rhs))
        for word in words:
            if not word in self.known and not word in self.unknown:
                if word in globals():
                    print('WARNING: {} in globals()'.format(word))
                    self.known.append(word)
                else:
                    try:
                        eval(word)
                        self.known.append(word)
                    except:
                        self.unknown.append(word)
        if verbose:
            print('  known:',self.known)
            print('unknown:',self.unknown)

    def process(self,max=10):
        """ Process the registered expressions """
        self.classify()
        for iter in range(max):
            ll=locals()
            if self.verbose:
                print('locals:',ll.keys())
                print('  known:',self.known)
                print('unknown:',self.unknown)
            if len(self.unknown)==0:
                print('RESULTS:')
                results={}
                for lhs in self.v.keys():
                    results[lhs]=eval(lhs)
                    print('  '+lhs+'={}'.format(results[lhs]))
                return results
            for lhs,rhs in self.v.items():
                if self.verbose:
                    print('lhs:',lhs)
                    print('rhs:',rhs)
                    if lhs in self.known:
                        print(' already known:',lhs)
                if lhs in self.unknown:
                    ok=True
                    for word in self.split(rhs):
                        ok=ok and word in self.known
                    if ok:
                        if word in self.pvars:
                            cmd="p.var('{}')".format(word)
                            print('  {}={}'.format(word,cmd))
                            ll[word]=eval(cmd)
                        try:
                            if self.verbose:
                                print('evaluating: '+lhs+'='+rhs)
                            ll[lhs]=eval(rhs)
                            self.known.append(lhs)
                            self.unknown.remove(lhs)
                            if self.verbose:
                                print('  appending to known:',lhs)
                        except:
                            if self.verbose:
                                print('  could not evaluate:',lhs)
        print('unknown:',self.unknown)

### Test

First with only scalars:

In [64]:
expressions=Expressions(p)
expressions.register('a=b+b')
expressions.register('b=c+1.5')
expressions.register('c=np.sqrt(4)')
expressions.process()

patch variables: ['d', 'e', 'et', 's', 'px', 'py', 'pz', 'bx', 'by', 'bz', 'qr', 'tt', 'phi', 'p1', 'p2', 'p3', 'b1', 'b2', 'b3', 'ux', 'uy', 'uz', 'u1', 'u2', 'u3', 'ee', 'E', 'rkap', 'src', 'tt', 'qr']
EXPRESSIONS:
  a=b+b
  b=c+1.5
  c=np.sqrt(4)
RESULTS:
  a=7.0
  b=3.5
  c=2.0


{'a': 7.0, 'b': 3.5, 'c': 2.0}

Now with ``d`` from a patch:

In [62]:
expressions=Expressions(p)
expressions.register('a=b+d')
expressions.register('b=c+1.5')
expressions.register('c=np.sqrt(4)')
expressions.process()

patch variables: ['d', 'e', 'et', 's', 'px', 'py', 'pz', 'bx', 'by', 'bz', 'qr', 'tt', 'phi', 'p1', 'p2', 'p3', 'b1', 'b2', 'b3', 'ux', 'uy', 'uz', 'u1', 'u2', 'u3', 'ee', 'E', 'rkap', 'src', 'tt', 'qr']
EXPRESSIONS:
  a=b+d
  b=c+1.5
  c=np.sqrt(4)
  d=p.var('d')
RESULTS:
  a=[[[ 3.6098187  3.6736176  3.7662528 ... 10.481551  11.525787  12.636501 ]
  [ 3.6093504  3.6726635  3.7647066 ... 10.578565  11.609745  12.693161 ]
  [ 3.6090527  3.6720753  3.763824  ... 10.664262  11.674124  12.727867 ]
  ...
  [ 3.5714664  3.6157455  3.6780894 ...  9.881198  10.643985  11.557012 ]
  [ 3.5666451  3.6086888  3.667572  ...  9.795718  10.587722  11.542337 ]
  [ 3.5620446  3.6019034  3.6580288 ...  9.767666  10.604405  11.60629  ]]

 [[ 3.6170611  3.6846864  3.7846773 ... 10.766865  11.780383  12.777465 ]
  [ 3.6171732  3.684478   3.7843893 ... 10.856383  11.859919  12.821154 ]
  [ 3.6174045  3.6845362  3.7845387 ... 10.929323  11.910524  12.833712 ]
  ...
  [ 3.5795927  3.6276758  3.6967642 ...  9

{'a': array([[[ 3.6098187,  3.6736176,  3.7662528, ..., 10.481551 ,
          11.525787 , 12.636501 ],
         [ 3.6093504,  3.6726635,  3.7647066, ..., 10.578565 ,
          11.609745 , 12.693161 ],
         [ 3.6090527,  3.6720753,  3.763824 , ..., 10.664262 ,
          11.674124 , 12.727867 ],
         ...,
         [ 3.5714664,  3.6157455,  3.6780894, ...,  9.881198 ,
          10.643985 , 11.557012 ],
         [ 3.5666451,  3.6086888,  3.667572 , ...,  9.795718 ,
          10.587722 , 11.542337 ],
         [ 3.5620446,  3.6019034,  3.6580288, ...,  9.767666 ,
          10.604405 , 11.60629  ]],
 
        [[ 3.6170611,  3.6846864,  3.7846773, ..., 10.766865 ,
          11.780383 , 12.777465 ],
         [ 3.6171732,  3.684478 ,  3.7843893, ..., 10.856383 ,
          11.859919 , 12.821154 ],
         [ 3.6174045,  3.6845362,  3.7845387, ..., 10.929323 ,
          11.910524 , 12.833712 ],
         ...,
         [ 3.5795927,  3.6276758,  3.6967642, ...,  9.987534 ,
          10.715347

Global variables are recognized, so delete any that should not be there:

In [48]:
del f

Setting a variable programmatically:

In [120]:
import sys
setattr(sys.modules[__name__],'ee',tmp)